### Orchestrator-Worker Workflow

Usecase:
- Generate a report on given topic, with each sub-topic having introduction and example.

In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain.chat_models import init_chat_model

from typing_extensions import TypedDict
from typing import List
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END



In [5]:
#base model

model = init_chat_model(model="groq:llama-3.1-8b-instant")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002353D2827B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000235661930B0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [7]:
#defining class for main topic and sub-topics
class Section(BaseModel):
    name: str = Field(description="Name for this section of the report")
    description: str = Field(description="GBrief overview of the main topics and concepts of the section")

class Sections(BaseModel):
    sections: List[Section]=Field(description="Sections of the report")


planner= model.with_structured_output(Sections)
    

#### Creating Workers Dynamically In Langgraph

Because orchestrator-worker workflows are common, LangGraph has the Send API to support this. It lets you dynamically create worker nodes and send each one a specific input. Each worker has its own state, and all worker outputs are written to a shared state key that is accessible to the orchestrator graph. This gives the orchestrator access to all worker output and allows it to synthesize them into a final output. 

Here we iterate over a list pf sections and Send each to a worker node.

In [ ]:
from langgraph.types import Send
from typing import Annotated
import operator
from operator import add

#Graph state

class State(BaseModel):
    topic:str #Report Topic
    sections: List[Section] #List of report sections
    completed_sections: Annotated[list, operator.add] # all workers write to this key parallely
    final_report: str #final report

#worker state

class WorkerState(TypedDict):
    section:Section
    completed_sections: Annotated[List, operator.add]


    

In [28]:
def subtopics_generation(state: TopicState):
    sub_topics = model.invoke(f"for a given topic, generate only sub topics in bullet points format, do not ellaborate them/\nTopic: {state['title']}")
    sub_topics = sub_topics.content.split("\n")
    return {"sub_topics": sub_topics}

def worker(state:TopicState):
    structured_model = model.with_structured_output(SubTopicState)
    info=""
    for point in state['sub_topics']:
        response = structured_model.invoke(f"generate notes on asked topic in defined structured format. Topic: {point} ")
        info = response.



SyntaxError: invalid syntax (4208119049.py, line 11)

In [ ]:

y

['• Types of Generative AI',
 '• Applications of Generative AI',
 '• Generative AI Models (GANs, VAEs, etc.)',
 '• Generative AI for Art and Design',
 '• Generative AI in Music and Audio',
 '• Generative AI in Text and Language',
 '• Generative AI for Data Augmentation',
 '• Generative AI in Healthcare',
 '• Generative AI for Product Generation',
 '• Generative AI Ethics and Bias',
 '• Generative AI Limitations and Challenges']

In [70]:
class actor(TypedDict):
    name:str
    gender:str

class movie(TypedDict):
    title:str
    actors:List[actor]    

In [72]:
s_model = model.with_structured_output(actor)


In [74]:
def node(state:movie):
    res= s_model.invoke(f"who is main leads in given movie {state['title']}")
    return {"actors": [res]}

graph = StateGraph(movie)

graph.add_node("node", node)

graph.add_edge(START, "node")

g = graph.compile()

g.invoke({"title":"Baahubali"})

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=actor>dict(name="Prabhas", gender="male")</function>\n'}}